# 02. 음성 Tool — STT & TTS (OpenAI Audio API)
**Day 4**

> OpenAI **Audio API** 한 곳에서 양방향 음성 처리를 합니다.
> - **STT**: `audio.transcriptions` — Whisper, 음성 → 텍스트
> - **TTS**: `audio.speech` — 텍스트 → 음성 MP3

회의록 자동화는 STT가 핵심이고, TTS는 **샘플 음성 생성·읽어주기·라운드트립 검증**에 씁니다.

---
## 0. 설치

In [1]:
!pip install openai python-dotenv

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

BASE = Path('.').resolve()
SAMPLES = BASE / 'samples'
SAMPLES.mkdir(exist_ok=True)

---
# Part A. STT — 음성 → 텍스트

`client.audio.transcriptions.create` (Whisper)

## 1. Whisper API 이해

| 파라미터 | 설명 |
|----------|------|
| `model` | `whisper-1` |
| `file` | mp3 / wav / m4a 등 |
| `language` | `ko` (언어 힌트) |

## 2. 샘플 MP3 준비

In [3]:
mp3_path = SAMPLES / 'meeting_sample.mp3'
print('파일 존재:', mp3_path.exists(), '|', mp3_path)

파일 존재: True | C:\Users\Admin\OneDrive\바탕 화면\실습\4일차\samples\meeting_sample.mp3


## 3. 기본 전사

In [4]:
#MP3 파일을 txt화
with open(mp3_path, 'rb') as f:
    result = client.audio.transcriptions.create( #chat -> audio
        model='whisper-1',
        file=f,
        language='ko',
    )

In [5]:
result.text

'안녕하세요. 마케팅 주간 스탠드업을 시작합니다. 이번 주 목표는 여름 캠페인 랜딩 페이지 오픈입니다. 수현님은 A, B 테스트 문구 3안을 수요일까지 준비해 주세요. 민재님은 광고 예산 상안을 오늘 중으로 조정해 주세요. 금요일 오전 10시에 선과 중간 리뷰를 진행합니다.'

In [6]:
with open(mp3_path, 'rb') as f:
    result = client.audio.translations.create(
        model='whisper-1',
        file=f,
    )

In [7]:
result.text

"Annyeonghaseyo. Marketing Weekly Stand-up is starting. This week's goal is to open the summer campaign landing page. Suhyeon, please prepare the A and B test phrases by Wednesday. Minjae, please adjust the ad budget to today. We will proceed with the election and mid-review at 10 a.m. on Friday."

---
# Part B. TTS — 텍스트 → 음성

`client.audio.speech.create`

In [8]:
SAMPLE_TEXT = (
    '안녕하세요. 오늘 실습을 시작합니다. '
    '오늘의 목표는 다양한 tool 생성과 RAG를 구현하는 것입니다.'
)

def text_to_speech(text, out_path, voice='nova', model='tts-1'):
    """OpenAI TTS로 텍스트를 MP3로 저장합니다."""
    out_path = Path(out_path)
    response = client.audio.speech.create(
        model=model,
        voice=voice,
        input=text,
        response_format='mp3',
    )
    out_path.write_bytes(response.content)
    return out_path

tts_path = SAMPLES / 'tts_openai_sample.mp3'
text_to_speech(SAMPLE_TEXT, tts_path)
print('저장:', tts_path)

저장: C:\Users\Admin\OneDrive\바탕 화면\실습\4일차\samples\tts_openai_sample.mp3


In [9]:
# Jupyter에서 바로 재생
from IPython.display import Audio, display
display(Audio(str(tts_path)))

## 5. voice / model 바꿔보기

In [10]:
for voice in ['alloy', 'nova', 'shimmer']:
    path = SAMPLES / f'tts_{voice}.mp3'
    text_to_speech('오늘 오후 실습 수업은 2시에 시작합니다.', path, voice=voice)
    print(voice, '→', path.name)

alloy → tts_alloy.mp3
nova → tts_nova.mp3
shimmer → tts_shimmer.mp3
